# KuroSiwo to VIIRS Showcase

This notebook demonstrates the minimal `src/` workflow for KuroSiwo-driven VIIRS extraction. It does three things:

1. Loads the KuroSiwo GeoPackage catalogue from `assets/ks_catalogue.gpkg`
2. Derives the per-event metadata table that Atlantis uses for event construction
3. Fetches VIIRS data for one KuroSiwo case and visualises the resulting rasters

Before running it, make sure you have:

- installed the notebook dependencies with `uv sync --extra notebooks`
- pulled the Git LFS assets with `git lfs pull`
- network access to the public NOAA JPSS S3 bucket at `https://noaa-jpss.s3.amazonaws.com`
- optional: access to `https://jpssflood.gmu.edu/downloads/pub` only if you want to force the legacy GMU backend

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists():
            return candidate
    raise RuntimeError('Run this notebook from inside the Atlantis repository.')


repo_root = find_repo_root(Path.cwd().resolve())
src_dir = repo_root / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

drafts_dir = repo_root / 'notebooks' / 'drafts'
scratch_dir = drafts_dir / 'data' / 'viirs_showcase'
scratch_dir.mkdir(parents=True, exist_ok=True)

catalogue_path = repo_root / 'assets' / 'ks_catalogue.gpkg'
selected_case = 'KuroSiwo_1111005'
days_before = 0
days_after = 0
viirs_backend = 'noaa_s3'
viirs_format = 'tif'
refresh_metadata_cache = False
metadata_cache_tag = f'{viirs_backend}_{viirs_format}_before_{days_before}_after_{days_after}'
metadata_output_path = scratch_dir / f'kurosiwo_metadata_{metadata_cache_tag}.csv'

print(f'repo_root: {repo_root}')
print(f'catalogue_path: {catalogue_path}')
print(f'scratch_dir: {scratch_dir}')
print(f'selected_case: {selected_case}')
print(f'viirs_backend: {viirs_backend}')
print(f'viirs_format: {viirs_format}')
print(f'metadata_output_path: {metadata_output_path}')
print(f'refresh_metadata_cache: {refresh_metadata_cache}')

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from shapely.geometry import box

from atlantis.fetchers.viirs import VIIRSFetcher
from atlantis.utils.kurosiwo import (
    build_kurosiwo_flood_events_from_catalogue,
    build_kurosiwo_flood_events_from_dataframe,
    derive_kurosiwo_metadata,
    load_kurosiwo_catalogue,
)

plt.style.use('seaborn-v0_8-whitegrid')
pd.options.display.max_columns = 20
pd.options.display.width = 140

## 1. Derive the Atlantis metadata table from the GeoPackage

This uses the production helper in `atlantis.utils.kurosiwo`, then enriches the metadata with two VIIRS-specific columns: `viirs_matches` and `viirs_available`.

The notebook caches that enriched metadata under `notebooks/drafts/data/viirs_showcase/` and reuses it on rerun unless `refresh_metadata_cache = True`. The cache filename is keyed by the selected VIIRS backend, format, and the `days_before` and `days_after` window.

In [ ]:
catalogue = load_kurosiwo_catalogue(catalogue_path)
metadata_required_columns = {
    'flood_case',
    'date_start',
    'date_end',
    'lat_min',
    'lat_max',
    'lon_min',
    'lon_max',
    'max_flood_extent_km2',
    'date_of_max_flood_extent',
    'viirs_matches',
    'viirs_available',
}

use_cached_metadata = metadata_output_path.exists() and not refresh_metadata_cache

if use_cached_metadata:
    metadata = pd.read_csv(metadata_output_path, parse_dates=['date_start', 'date_end'])
    missing_columns = metadata_required_columns - set(metadata.columns)
    if missing_columns:
        print(
            'Cached metadata is missing required columns: '
            + ', '.join(sorted(missing_columns))
            + '. Rebuilding cache.'
        )
        use_cached_metadata = False

if use_cached_metadata:
    metadata['viirs_matches'] = pd.to_numeric(metadata['viirs_matches'], errors='coerce').fillna(0).astype(int)
    metadata['viirs_available'] = (
        metadata['viirs_available']
        .astype(str)
        .str.lower()
        .map({'true': True, 'false': False})
        .fillna(False)
        .astype(bool)
    )
    cache_status = 'loaded from cache'
else:
    metadata = derive_kurosiwo_metadata(catalogue_path)
    fetcher = VIIRSFetcher(backend=viirs_backend, data_format=viirs_format)

    availability_rows = []
    events = build_kurosiwo_flood_events_from_dataframe(
        metadata,
        days_before=days_before,
        days_after=days_after,
    )

    for event in events:
        search_results = fetcher.search(event)
        availability_rows.append(
            {
                'flood_case': event.event_id,
                'viirs_matches': len(search_results),
                'viirs_available': bool(search_results),
            }
        )

    availability = pd.DataFrame(availability_rows)
    metadata = metadata.merge(availability, on='flood_case', how='left')
    metadata['viirs_matches'] = metadata['viirs_matches'].fillna(0).astype(int)
    metadata['viirs_available'] = metadata['viirs_available'].fillna(False).astype(bool)
    metadata.to_csv(metadata_output_path, index=False)
    cache_status = 'rebuilt and cached'

print(f'catalogue rows: {len(catalogue):,}')
print(f'distinct events: {catalogue["actid"].nunique()}')
print(f'metadata cache: {metadata_output_path.relative_to(repo_root)} ({cache_status})')
metadata.head()

In [ ]:
top_events = metadata.nlargest(10, 'max_flood_extent_km2').sort_values('max_flood_extent_km2')

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top_events['flood_case'], top_events['max_flood_extent_km2'], color='#2f6c8f')
ax.set_title('Largest KuroSiwo events by derived flood extent')
ax.set_xlabel('max_flood_extent_km2')
ax.set_ylabel('')
plt.show()

## 2. Inspect one KuroSiwo case

Choose a case where `viirs_available` is `True` if you want the fetch step to succeed with the current date window.

In [ ]:
selected_event = metadata.loc[metadata['flood_case'] == selected_case].iloc[0]
selected_actid = int(selected_case.split('_')[-1])
event_rows = catalogue.to_crs('EPSG:4326')
event_rows = event_rows[event_rows['actid'] == selected_actid].copy()
event_bbox = box(
    selected_event['lon_min'],
    selected_event['lat_min'],
    selected_event['lon_max'],
    selected_event['lat_max'],
)

selected_event[
    ['flood_case', 'date_start', 'date_end', 'max_flood_extent_km2', 'viirs_matches', 'viirs_available']
]

## 3. Build the Atlantis event and fetch VIIRS

This goes through the same `FloodEvent` construction path used by the CLI. By default Atlantis now searches the NOAA JPSS public S3 bucket and stores the downloaded TIFFs and processed rasters under `notebooks/drafts/data/viirs_showcase/`. Set `viirs_backend = 'gmu_legacy'` above if you want to compare against the older GMU source.

In [ ]:
selected_event = metadata.loc[metadata['flood_case'] == selected_case].iloc[0]
if not bool(selected_event['viirs_available']):
    available_cases = metadata.loc[metadata['viirs_available'], 'flood_case'].tolist()
    preview = ', '.join(available_cases[:10])
    raise RuntimeError(
        f'{selected_case} has no VIIRS matches for the configured window. '
        f'Choose a case where viirs_available is True, for example: {preview}'
    )

events = build_kurosiwo_flood_events_from_catalogue(
    catalogue_path,
    case=selected_case,
    days_before=days_before,
    days_after=days_after,
)
event = events[0]

In [ ]:
fetch_dir = scratch_dir / selected_case / 'viirs'

fetcher = VIIRSFetcher(backend=viirs_backend, data_format=viirs_format)
print(f'Using VIIRS backend={fetcher.backend}, format={fetcher.data_format}, base_url={fetcher.base_url}')

In [ ]:
results = fetcher.fetch(event, fetch_dir)
if not results:
    raise RuntimeError('No VIIRS results returned. Check network access and archive availability.')

print(f'event window: {event.start_date} -> {event.end_date}')
for result in results:
    print(f'{result.event_id}: {len(result.files)} files from {fetcher.backend}/{fetcher.data_format}')
    for path in result.files:
        print(f'  - {path.relative_to(repo_root)}')

## 4. Visualise the fetched VIIRS layers

The fetcher writes three rasters per date: `flood_extent`, `quality_mask`, and `permanent_water`.

In [ ]:
dataset = fetcher.to_dataset(results[0])
dataset

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
dataset['flood_extent'].plot(ax=axes[0], cmap='Blues')
axes[0].set_title('Flood extent')
dataset['quality_mask'].plot(ax=axes[1], cmap='gray')
axes[1].set_title('Quality mask')
dataset['permanent_water'].plot(ax=axes[2], cmap='viridis')
axes[2].set_title('Permanent water')
for ax in axes:
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
plt.show()

## 5. CLI equivalents

The same steps can be run outside the notebook with:

```bash
uv run python -m atlantis.cli build-kurosiwo-metadata --catalogue assets/ks_catalogue.gpkg
uv run python -m atlantis.cli fetch-kurosiwo-viirs --catalogue assets/ks_catalogue.gpkg --case KuroSiwo_470 --days-before 0 --days-after 0 --viirs-backend noaa_s3 --viirs-format tif
```

To force the legacy GMU source instead, add `--viirs-backend gmu_legacy`. The notebook-specific `viirs_available` and `viirs_matches` columns are computed here by querying the configured VIIRS archive for each derived event window.